Q3: How do we find informed players in this trading market?
- Their trade size is unusually large relative to the population of trades
- Their trade size is large relative to their own history (if we have more player_id histories, and we see a player_id ABC did a trade that was much larger than their history, or they have no history, and they just did a trade that was +4 stdev bigger than average quantities in the past few hours / days / weeks, that's a sign).  But we don't have it - player_id and player name are NA here.
- They hit through the stack - they trade a price that is 2-3 cents through the best bid (sell 2-3 cents below best bid) or best ask (buy 2-3 cents above best bid)
- There might be some anomaly detection that we can do (use machine learning or anomaly detection algorithms on the features of price and quantity, especially relative to the order book?)
- Also, if they do a trade like this, and then the market moves a few min or hours after this, it means that they knew the info first, and then it leaked to other participants later.  This doesn't help us on that trade, but it helps us know that trader is an informed player so we should watch his trades going forward
- How do identify trades in real-time?  When the stack moves in a magnitude that is bigger than what it has historically done in that market, or if the trade size is unusually large compared to other trades in the market

- Another way to identify informed players is that their trades happen first, and then the market moves heavily in their favor.  So for example, we once again walk through the timeline, from start of the dataset to the end of the dataset, and every time we encounter a trade in the trades dataset, we analyze what happened in that market in the following minutes (we can pick any time frame, such as 10 minutes or 30 minutes afterward), and flag cases where the market moved in the same direction as that trader, especially if the trade did a large-sized trade
- The issue with this metric is that you cannot know that the trade was informed until after the market already moved in his favor, and likely against yours as a market-maker.  In terms of what you are given in real-time, the only thing that you can know about that trade is its price and size

- If you actually had more data on the players, such as the player name, and player_id, and you saw that they did an informed trade, then you can watch for their trades going forward.  Let's say that you saw player ABC sold p in a market at p = 0.6, and then 5 min later, p crashed down to 0.55.  Then you know that ABC probably has some special edge in the market (private non public information maybe), so you watch next time a trade from ABC happens.  Unfortunately, based on the data exploration that we did earlier, all of the columns for player name and player ID are NA in this dataset, so we can't use it anyway, if the data is in this format

Response to Claude:
- Agreed that we need to look at both books and trades datasets
- The most informed action is probably one where the size is unusually large, and it happens in a short time span, perhaps over multiple markets.  I also bet that aggressive_buy_flag is going to be True, because someone who knows that the event will resolve at either $0 or $1 will not care about paying up an extra $0.01 or $0.02 in bid/ask spread
- I would check this in the trades dataset first, and then the order book data set second
- Proposal: As before, let's walk through time, from start of the 6 hours to the end of the hours.  Let's look at every trade as we go along forward in time.  We keep a running record of the trades as they happen.  We can create a dataframe of trades, with columns = market, price, quantity, direction (aggressor_buy_flag = True is buy, False is sell), and timestamp.  Whenever we see a trade that is beyond let's say 3 standard deviations away from the average trade size, and especially if the price compares to the order book and it is beyond the best_bid or above best_ask, we flag it (the person is hitting through multiple prices in the stack)


- In terms of the risk, we can quantify the exact dollar risk by looking at the direction of the trade and the difference between the entry level and 0 vs. 1.  For instance, if someone bought 100 contracts at 0.7, the worst thing that could happen would be if the event resolved at 0, so they lost 0.7 on 100 contracts, which is a loss of $70 for instance.
- Beyond just the financial risk of monetary losses, this also risks leaking information into the market for other players - someone who is watching for such trades will get the information that someone informed made a bet that has a higher than normal chance of paying out. 
- There are also other issues like legal ramifications. Examples:

Regulators have made clear that trading event contracts on MNPI (material non-public information) can be prosecuted even though these aren't traditional securities. The CFTC's Enforcement Division put out a Prediction Markets Advisory in February 2026, publicly describing MNPI-based event-contract trading as conduct the CFTC can pursue as "insider trading" under CEA § 6(c)(1) and Rule 180.1. The legal theory used is misappropriation of confidential information, not classic securities insider trading, because event contracts fall under commodities law. 
Snell & Wilmer

Actual cases so far
- The YouTube editor case (Kalshi, 2025) — a trader who was a YouTube channel editor had advanced knowledge of video contents before they posted, and traded on that. Kalshi hit him with a $20,397.58 penalty (disgorgement plus fine) and a 2-year exchange suspension. 
Commodity Futures Trading Commission
Political candidate trading on his own race (Kalshi, May 2025) — a political candidate was found trading on his own candidacy, which Kalshi flagged and disciplined.
- Michele Spagnuolo / Google engineer (May 2026) — the CFTC and DOJ charged a Google employee with using material nonpublic information to trade Polymarket contracts tied to the browser's "Year in Search" lists — first case involving a private-sector company employee trading on internal work knowledge. 

### Plan for Q3 -- to review and revise before any code is written

#### What the question asks for

> What is the most informed action / series of actions in the dataset? For the person that
> made such an action, what was an "economic cost or risk" that they exposed themselves to
> while making this action, or how might this action have ended poorly (beyond simply
> losing more money if they lost the trade)?

Two deliverables, and the second is the one that is not a standard screen:

1. **Name a specific action or series of actions** -- an actual timestamp, market(s), size
   and direction out of this dataset, not a method for finding them.
2. **Say what it cost the person to do it, beyond the money at risk on the trade.** The
   prompt explicitly rules out "they might lose more money", so max-loss arithmetic does
   not answer it. It is about the second-order consequences of having to show your hand to
   the market in order to get the position on.

#### What the data supports

| column | use |
| --- | --- |
| `aggressor_buy_flag` | which side **initiated** the trade. Verified against the book: True -> price is the best ask 94.9% of the time (initiator lifted the offer, bought YES); False -> price is the best bid 95.1% (initiator hit the bid, sold YES). |
| `qty`, `price`, `recv_ts_utc` | size, level, time |
| `native_id` | market, and via `parse_chain_and_strike` the chain and strike |
| `exchange_trade_id` | unique per trade -- usable as a key, but links nothing across trades |
| `player_id`, `player_name` | **100% null** -- no trader identity, no per-trader history, no "watch this account going forward" |

Two things this rules out. There is no way to track an account across trades, which kills
the strongest real-world approach. And `aggressor_buy_flag` does **not** separate makers
from takers -- every trade has one of each by definition -- so we cannot filter for
"aggressive" trades. Aggression has to be measured as sweeping multiple price levels
instead, which is rare: only 44 bursts in the whole dataset cross more than one level.

Every market is an "Over": `outcome_name` takes only the values `Over`,
`Milwaukee Brewers;Over`, `Pittsburgh Pirates;Over`. So buying YES anywhere means "more
runs", and direction aligns across all chains with no sign-flipping.

The tape is lopsided: 1,704 initiated sells against 528 initiated buys, roughly 76/24.

#### The core idea: score the basket, not the market

An informed trader with a view on run scoring does not express it in one market. They
spray across strikes and across correlated chains. Scoring each market against its own
size distribution would therefore miss exactly the pattern we are hunting -- six 500-lot
trades spread over six strikes look unremarkable individually and are one 3,000-lot action
in reality.

So the unit of analysis is a **basket of similar trades in the same direction inside a
short window**, and the reference distribution is built from those basket totals.

#### Defining "similar": three islands

The 33 markets split into three groups by which random variable they track:

| island | chains | strikes | trades | contracts |
| --- | --- | --- | --- | --- |
| **Game runs** | RFI, F5TOTAL, TOTAL | 1 + 7 + 11 = 19 | 2,084 | 466,154 |
| MIL runs | TEAMTOTAL-MIL | 7 | 121 | 14,681 |
| PIT runs | TEAMTOTAL-PIT | 7 | 27 | 1,270 |

Within the game-runs island, everything counts as similar to everything else. RFI is runs
in the 1st inning, F5TOTAL is runs through 5 innings, TOTAL is runs in the whole game --
all three are the same quantity measured over nested windows, and Q1 established the
pathwise nesting RFI within F5 within TOTAL. A trader who thinks this game will be
high-scoring can express that view in any of the 19 markets, so a burst across them is one
action.

There are **no links between islands**. MIL and PIT are different teams and different
random variables, and keeping the islands disjoint is what makes "family" a clean partition
rather than a relation that would transitively drag MIL and PIT together through TOTAL.

**MIL and PIT are then dropped from the analysis entirely.** They are not liquid enough for
an informed trader to get size on -- PIT traded 1,270 contracts across 27 trades in six
hours, MIL 14,681 across 121 -- so an informed trader could not make meaningful money there
even with a correct view. There is also far too little data to establish what an outlier
even looks like in those chains. Removing them costs us nothing and avoids reporting a
"finding" resting on 27 observations.

That leaves the game-runs island: **2,084 trades, 466,154 contracts, 19 markets.**

#### Step 1 -- bucket the tape into 1-second intervals

Cut the timeline into consecutive, non-overlapping 1-second buckets, and within each
bucket sum quantity separately for each direction. Each bucket-direction pair becomes one
observation.

Non-overlapping matters. A sliding 1-second look-back anchored on every trade would let one
event produce one observation per print: the 23:32:07 event is 39 prints in a single
second, so it would appear 39 times, filling the leaderboard with copies of itself and
stuffing the reference distribution with 39 near-duplicates of the outlier we are trying to
detect. Consecutive buckets give one event, one row.

Checked against gap-based clustering (break when more than 1 second passes with no
same-direction trade) and the two agree closely -- 307 vs 303 buy observations, 925 vs 851
sell observations, identical maxima -- and the largest event does not straddle a bucket
boundary. Fixed buckets are simpler, so we use those.

Result: roughly 1,232 observations, 307 buy-side and 925 sell-side.

#### Step 2 -- score each bucket against the running distribution

For each bucket, compare its total quantity against the distribution of all prior bucket
totals in the same direction. Using only prior buckets keeps the score honest as a
real-time statistic, which is what Q4a will need.

The fence is the Tukey rule, `Q3 + 1.5 * IQR`, on that running distribution. Median and
standard deviation are both unusable here: the distribution is extremely fat-tailed (trade
mean 216, standard deviation 1,102, max 27,430), so the large baskets we are hunting
inflate the mean and the standard deviation themselves, raising the bar and hiding the next
one. Quartiles are resistant to exactly that.

The fence is a **screen, not the answer**. Having passed it, buckets are ranked by size
relative to the running median, so the answer is "this basket was N times the typical
basket" rather than a raw contract count.

Warm-up: the first stretch of the session has too little history to score against, so those
buckets are reported as unscoreable rather than given a misleading score.

#### Step 3 -- markout, to separate informed from merely large

Large is not the same as informed. Using `aggressor_buy_flag` for direction, measure how far
the market moved in the initiator's favour after the action:

    markout = (mid_after - mid_at_action) * (+1 if initiator bought else -1)

at +1, +5 and +30 minutes. Positive markout means the market moved their way. This is the
retrospective confirmation that a basket was informed rather than just big -- and the fact
that it can only be known after the fact is itself worth saying in the write-up, since it is
precisely the gap Q4a has to close.

Two caveats inherited from Q1 and Q2: the mid must come from a quote that is genuinely
fresh at the measurement point rather than a stale one carried forward, and 66% of trades
land inside book-feed gaps of more than 5 seconds, so some markout windows will be
unmeasurable and should be reported as missing rather than filled.

#### Step 4 -- name the winner and write the risk section

Leading candidate on the evidence so far: **23:32:07 UTC**, 37 initiated sells totalling
61,241 contracts across RFI and F5TOTAL-4, roughly 8 minutes before first pitch. Next
largest is 41,997 at 21:28:26 on TOTAL-8. Step 3 either confirms it or replaces it -- the
markout decides, not the size.

For whichever basket wins, the economic cost / risk section covers:

1. **Information leakage.** The trade tells everyone watching the tape that someone with
   conviction has taken a side. The edge is worth less the moment it is exercised.
2. **The position cannot be exited.** A 61k basket is a large fraction of everything these
   markets traded in six hours. There is no way out at a sensible price, so the trader is
   locked in to settlement. That converts a view about probability into an all-or-nothing
   outcome at $0 or $1 -- they cannot take the win at 0.80 and go home.
3. **The market maker widens or pulls.** Q1 established that one maker appears to quote the
   whole chain off a single fitted distribution. A large take tells that maker they have
   been adversely selected, so they widen or step away. The trader burns the edge on the
   first clip and cannot repeat it.
4. **The trade telegraphs across the whole ladder.** Because the chain is internally
   consistent -- Q1 found zero arbitrage violations anywhere -- repricing one strike forces
   the maker to reprice every correlated strike. Hitting one market leaks the view into
   eighteen others simultaneously, revealing far more than intended.
5. **Legal and regulatory exposure**, per the CFTC advisory and cases written up above: if
   the edge came from material non-public information, the downside is not a losing trade
   but disgorgement, fines, exchange suspension, or prosecution.

The max-loss arithmetic (100 contracts at 0.70 risks $70) stays as a single framing
sentence, since the prompt explicitly excludes it as the answer.

#### Open questions still to settle

1. **Is 1 second the right bucket width?** It is a round number, not a measured one. Worth
   checking whether the answer changes at 0.5s or 2s before committing.
2. **Which markout horizon decides the winner** if +1, +5 and +30 minutes disagree? Better
   to fix this in advance than to pick the one that flatters a preferred answer afterwards.
3. **How much history is enough** before a bucket is scoreable? Needs a stated minimum.